# MegaDetector Setup (third-party/eb_MegaDetector_v6)

This notebook provides a step-by-step setup path for the local MegaDetector v6
dependency under:

- `third-party/eb_MegaDetector_v6`

MegaDetector v6 is packaged as the `megadetector_core` Python package (a thin
wrapper over [PyTorch-Wildlife](https://github.com/microsoft/CameraTraps)). It is
intended for initial bootstrap and repeatable environment validation.

## Step 1 — Confirm repository root and MegaDetector path

In [8]:
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
md_path = repo_root / "third-party" / "eb_MegaDetector_v6"

print(f"Repository root: {repo_root}")
print(f"MegaDetector path: {md_path}")
print(f"Exists: {md_path.exists()}")

Repository root: /Users/elhorte/git/ebio/project-id
MegaDetector path: /Users/elhorte/git/ebio/project-id/third-party/eb_MegaDetector_v6
Exists: True


## Step 2 — Clone Earth-Biometrics MegaDetector fork if missing

Run the next cell only if `third-party/eb_MegaDetector_v6` does not exist.

Source repo:
- `git@github.com:Earth-Biometrics/eb_MegaDetector_v6.git`

In [9]:
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
third_party_dir = repo_root / "third-party"
md_dir = third_party_dir / "eb_MegaDetector_v6"
repo_url = "git@github.com:Earth-Biometrics/eb_MegaDetector_v6.git"

third_party_dir.mkdir(parents=True, exist_ok=True)
if md_dir.exists():
    print(f"MegaDetector already present at {md_dir}")
else:
    subprocess.run(["git", "clone", repo_url, str(md_dir)], check=True)
    print(f"Cloned {repo_url} to {md_dir}")

MegaDetector already present at /Users/elhorte/git/ebio/project-id/third-party/eb_MegaDetector_v6


## Step 3 — Install MegaDetector v6 (editable)

MegaDetector v6 is installed by pip-installing the vendored checkout in editable
mode. This installs the `megadetector_core` / `megadetector_ai` packages and pulls
in the runtime dependencies (PytorchWildlife, ultralytics, torch, …) declared in
`pyproject.toml`.

This cell also installs `soundfile` and `librosa`: PyTorch-Wildlife imports these
audio libraries at package-load time (for its bioacoustics module) even when you
only use image detection, but does not declare them as dependencies.

> Editable installs register their path in a `.pth` file that Python only reads at
> interpreter startup, so a kernel that is already running won't see
> `megadetector_ai` until it restarts. Step 4 (and notebook 02) work around this by
> adding the package's `src/` folder to `sys.path` at runtime, so a kernel restart
> is **not required** — but restarting is still the cleanest option if you prefer.

> Note: the repo's `requirements.txt` only lists documentation tooling; the actual
> runtime dependencies live in `pyproject.toml`, so we install with `pip install -e`.

In [10]:
from pathlib import Path
import subprocess
import sys

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
md_root = repo_root / "third-party" / "eb_MegaDetector_v6"

if not md_root.exists():
    raise FileNotFoundError(f"MegaDetector repo not found: {md_root}. Run Step 2 first.")
if not (md_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        f"pyproject.toml not found in {md_root}; expected a MegaDetector v6 checkout."
    )

# Editable install of MegaDetector v6 (megadetector_core / megadetector_ai).
cmd = [sys.executable, "-m", "pip", "install", "-e", str(md_root)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

# PyTorch-Wildlife imports these audio libraries at package-load time (bioacoustics
# module) even for image-only detection, but does not declare them as deps.
audio_deps = ["soundfile", "librosa"]
cmd = [sys.executable, "-m", "pip", "install", *audio_deps]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\nMegaDetector v6 installed in editable mode.")
print("!! RESTART THE KERNEL now (Kernel -> Restart) before running Step 4, so the")
print("   editable install's path entry is picked up.")

Running: /Users/elhorte/git/ebio/project-id/.venv/bin/python -m pip install -e /Users/elhorte/git/ebio/project-id/third-party/eb_MegaDetector_v6
Obtaining file:///Users/elhorte/git/ebio/project-id/third-party/eb_MegaDetector_v6
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for megadetector-core (pyproject.toml): started
  Building editable for megadetector-core (pyproject.toml): finished with status 'done'
  Created wheel for megadetector-core: filename=megadetector_core-0.1.0-0.editable-py3-none-any.whl size=12050 sha256=85

## Step 4 — Smoke test imports

This verifies that MegaDetector v6 and its core dependencies import cleanly.

In [11]:
import sys
import subprocess
from pathlib import Path

# MegaDetector v6 is installed editable; its path is registered in a .pth file
# that Python only reads at interpreter startup, so a kernel that was running
# during Step 3's install can't see `megadetector_ai` until it restarts. Adding
# the package's src/ directory to sys.path makes the import work without a restart.
_md_src = (
    Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
    / "third-party" / "eb_MegaDetector_v6" / "src"
)
if _md_src.is_dir() and str(_md_src) not in sys.path:
    sys.path.insert(0, str(_md_src))

import importlib

modules = ["torch", "cv2", "PIL", "numpy", "soundfile", "librosa", "PytorchWildlife", "megadetector_ai"]
results = {}

for name in modules:
    try:
        importlib.import_module(name)
        results[name] = "OK"
    except Exception as exc:
        results[name] = f"FAILED: {exc}"

# Confirm the v6 detector class is importable.
try:
    from megadetector_ai import MegaDetectorV6
    results["MegaDetectorV6"] = "OK"
except Exception as exc:
    results["MegaDetectorV6"] = f"FAILED: {exc}"

results

{'torch': 'OK',
 'cv2': 'OK',
 'PIL': 'OK',
 'numpy': 'OK',
 'soundfile': 'OK',
 'librosa': 'OK',
 'PytorchWildlife': 'OK',
 'megadetector_ai': 'OK',
 'MegaDetectorV6': 'OK'}

## Step 5 — Next action

After successful setup, run:

- [02_megadetector_local_images.ipynb](02_megadetector_local_images.ipynb)

to exercise MegaDetector on local images.